In [1]:
%%capture
!pip install parler-tts
!pip install "protobuf>=5.28.0" --upgrade
# !pip install parler-tts transformers

In [2]:
"""
Nepali Text-to-Speech using Indic Parler-TTS
Speaker: Amrita (only recommended Nepali speaker)
"""

from google.colab import userdata
from huggingface_hub import login
from parler_tts import ParlerTTSForConditionalGeneration
from transformers import AutoTokenizer
import torch, numpy as np, soundfile as sf
import IPython.display as ipd

MODEL_ID = "milanakdj/indic-parler-tts-nepali-finetuned-dgx-v9-cosine"
DEVICE = "cuda:0" if torch.cuda.is_available() else "cpu"

# Load model and tokenizers
model = ParlerTTSForConditionalGeneration.from_pretrained(MODEL_ID).to(DEVICE)
prompt_tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
description_tokenizer = AutoTokenizer.from_pretrained(model.config.text_encoder._name_or_path)


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/3.75G [00:00<?, ?B/s]

  "_name_or_path": "google/flan-t5-large",
  "architectures": [
    "T5ForConditionalGeneration"
  ],
  "classifier_dropout": 0.0,
  "d_ff": 2816,
  "d_kv": 64,
  "d_model": 1024,
  "decoder_start_token_id": 0,
  "dense_act_fn": "gelu_new",
  "dropout_rate": 0.1,
  "eos_token_id": 1,
  "feed_forward_proj": "gated-gelu",
  "initializer_factor": 1.0,
  "is_encoder_decoder": true,
  "is_gated_act": true,
  "layer_norm_epsilon": 1e-06,
  "model_type": "t5",
  "n_positions": 512,
  "num_decoder_layers": 24,
  "num_heads": 16,
  "num_layers": 24,
  "output_past": true,
  "pad_token_id": 0,
  "relative_attention_max_distance": 128,
  "relative_attention_num_buckets": 32,
  "tie_word_embeddings": false,
  "transformers_version": "4.46.1",
  "use_cache": true,
  "vocab_size": 32128
}

  "_name_or_path": "ylacombe/dac_44khz",
  "architectures": [
    "DacModel"
  ],
  "codebook_dim": 8,
  "codebook_loss_weight": 1.0,
  "codebook_size": 1024,
  "commitment_loss_weight": 0.25,
  "decoder_hidden_si

generation_config.json:   0%|          | 0.00/218 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/990 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/552 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

In [3]:

# Your Nepali text here
prompt = "नमस्ते, तपाईंलाई कस्तो छ?"

description = "Amrita speaks with a high pitch at a slow pace. Her voice is clear, with excellent recording quality and very clear audio."

# Tokenize
desc_inputs = description_tokenizer(description, return_tensors="pt").to(DEVICE)
prompt_inputs = prompt_tokenizer(prompt, return_tensors="pt").to(DEVICE)

# Generate
with torch.inference_mode():
    generation = model.generate(
        input_ids=desc_inputs.input_ids,
        attention_mask=desc_inputs.attention_mask,
        prompt_input_ids=prompt_inputs.input_ids,
        prompt_attention_mask=prompt_inputs.attention_mask,
        max_new_tokens=1000,
    )

audio = generation.cpu().numpy().squeeze()
sf.write("nepali_output.wav", audio, model.config.sampling_rate)
print("Saved to nepali_output.wav")

Saved to nepali_output.wav


In [4]:
TRAINING_DESCRIPTIONS = [
    "Amrita speaks with a deep, formal Nepali voice. Her speech is clear, steady and authoritative with natural pacing in a quiet noise-free environment.",
]

sentences = [
    "नमस्ते, तपाईंलाई आज कस्तो सहयोग चाहिन्छ?",
    "म तपाईंको सहायक बोल्दैछु।",
    "कृपया आफ्नो समस्या विस्तारमा बताइदिनुहोस्।",
    "आजको मौसम निकै राम्रो देखिन्छ।",
    "तपाईंको दिन शुभ रहोस्।",
    "म नेपाली भाषामा पनि कुरा गर्न सक्छु।",
    "के तपाईंलाई कुनै जानकारी चाहिएको छ?",
    "तपाईंले पठाएको अनुरोध प्रक्रिया हुँदैछ।",
    "कृपया केही क्षण प्रतीक्षा गर्नुहोस्।",
    "धन्यवाद, तपाईंको सन्देश प्राप्त भयो।",
    "यो एउटा परीक्षण वाक्य हो।",
    "कम्प्युटर विज्ञान निकै रोचक विषय हो।",
    "म नयाँ प्रविधिहरू सिक्दैछु।",
    "नेपाल प्राकृतिक सौन्दर्यले भरिएको देश हो।",
    "काठमाडौं नेपालको राजधानी शहर हो।",
    "आज तपाईंले के सिक्नुभयो?",
    "संगीत सुन्न मलाई मन पर्छ।",
    "कृत्रिम बुद्धिमत्ता भविष्यको महत्वपूर्ण प्रविधि हो।",
    "तपाईंको इन्टरनेट जडान स्थिर देखिन्छ।",
    "कृपया फेरि प्रयास गर्नुहोस्।",
    "यो आवाज परीक्षणको लागि प्रयोग गरिएको वाक्य हो।",
    "विद्यालयमा विद्यार्थीहरू अध्ययन गर्दैछन्।",
    "हामी नयाँ परियोजनामा काम गरिरहेका छौं।",
    "तपाईंको फाइल सफलतापूर्वक अपलोड भयो।",
    "अब म अर्को वाक्य पढ्दैछु।",
    "तपाईंलाई सहयोग गर्न पाउँदा खुशी लाग्यो।",
    "सुरक्षित यात्रा गर्नुहोस्।",
    "तपाईंको अर्डर तयार हुँदैछ।",
    "कृपया आफ्नो नाम भन्नुहोस्।",
    "यो प्रणाली अहिले सक्रिय अवस्थामा छ।"
]

# Encode the description once outside the loop (it does not change)
description = TRAINING_DESCRIPTIONS[0]
desc_inputs = description_tokenizer(description, return_tensors="pt").to(DEVICE)

for i, sentence in enumerate(sentences):
    prompt_inputs = prompt_tokenizer(sentence, return_tensors="pt").to(DEVICE)

    with torch.inference_mode():
        gen = model.generate(
            input_ids=desc_inputs.input_ids,
            attention_mask=desc_inputs.attention_mask,
            prompt_input_ids=prompt_inputs.input_ids,
            prompt_attention_mask=prompt_inputs.attention_mask,
            max_new_tokens=1000,
        )

    audio = gen.cpu().numpy().squeeze().astype(np.float32)
    max_val = np.abs(audio).max()
    if max_val > 1e-6:
        audio = audio / max_val

    print(f"\n[Sentence {i+1}] {sentence}")
    ipd.display(ipd.Audio(audio, rate=model.config.sampling_rate))


[Sentence 1] नमस्ते, तपाईंलाई आज कस्तो सहयोग चाहिन्छ?



[Sentence 2] म तपाईंको सहायक बोल्दैछु।



[Sentence 3] कृपया आफ्नो समस्या विस्तारमा बताइदिनुहोस्।



[Sentence 4] आजको मौसम निकै राम्रो देखिन्छ।



[Sentence 5] तपाईंको दिन शुभ रहोस्।


...


[Sentence 26] तपाईंलाई सहयोग गर्न पाउँदा खुशी लाग्यो।



[Sentence 27] सुरक्षित यात्रा गर्नुहोस्।



[Sentence 28] तपाईंको अर्डर तयार हुँदैछ।



[Sentence 29] कृपया आफ्नो नाम भन्नुहोस्।



[Sentence 30] यो प्रणाली अहिले सक्रिय अवस्थामा छ।
